In [130]:
#i/o imports, pandas
from pathlib import Path
from natsort import natsorted
from typing import List
import pandas as pd

#rdkit imports
import rdkit
from rdkit import Chem

"""
### 'CSV_combiner.ipynb' - @GCH v1.0 - last updt. 04/17/2026

### Notebook Overview:
This notebook contains code to combine and deduplicate SMILES data from multiple incoming .csvs containing a column
of 'SMILES' data. The general flow of the program is as follows:
1. Search specified directory for .csv files containing a column named "SMILES" (any capitalization)
2. From the list of all SMILES, try to convert each SMILE to an RDKit Mol object; omit failed SMILES
3. Convert deduplicated SMILES into Canonical SMILES using RDKit
4. Peform string-based deduplication using Canonical SMILES
5. Output a deduplicated .csv of SMILES and their data source/ID info to a specified output directory

### Motivation:
A simple and expandable script to parse .csv files for 'SMILES' data / simple deduplication

### How to Use this Notebook: 
1. Define relevant Paths to output files in the PATH cell below
2. Run all cells in the notebook

"""

In [131]:
### Define PATHS/relevant directories ### 
script_dir = Path.cwd() #path containing the current script

#the project TLD is 1 "parent" above current dir
project_tld = script_dir.parent

#target dir containing .csv's to parse for SMILES data
input_csv_dir = project_tld / "csv_backups" / "input_csv_datasets"
input_csv_dir.mkdir(exist_ok=True, parents=True)

In [132]:
def get_filepaths_in_target_dir(target_directory: Path, extension: str, printing=True) -> Path:
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches for specific .ext
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths - a list of filepaths for each file wit .extension' in 'target_directory'
    """
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.glob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{target_directory.stem}/\033[0m'")

    #return the list of located filepaths
    return filepaths

In [133]:
def parse_csvs_for_smiles (target_directory: Path) -> pd.DataFrame:
    """
    Function: Combine cols of SMILES data from multiple .csv files, if present.
        - Parses /target_directory/ for any files with the '.csv' extension
        - Ensures they have some semblance of a 'SMILES' column header
        - concatenates the SMILES columns, retaining the original files' IDs, if present, and origin dataset

    Input:
        - target_directory: Path; a Path to directory containing .csv data
        
    Returns:
        - combined_df; pd.DataFrame - a pd dataframe (df) with column format:
           data_source  |   regid    |     smiles
           (origin.csv)  (origin ID)   (SMILES string)
    """
    
    #print a notice to user about what's happening where
    print(f"\033[1mParsing targeted '/{target_directory.name}/' for .csv files containing SMILES data...\033[0m\n")

    #parse targeted directory for .csv files 
    csvs_to_check = get_filepaths_in_target_dir(target_directory, '.csv', printing=False)

    #loop through each csv and find the ID/SMILES column in each 
    list_of_smiles_nums = []
    for csv_file in csvs_to_check:
        
        ### Read in CSV data as pd dataframe ###
        source_name = (f"{csv_file}")
        #convert csv into pandas dataframe for manip
        incoming_df = pd.read_csv(source_name)
        #make all the column headers lowercase for consistent parsing/finding correct columns
        incoming_df.columns = [x.lower() for x in incoming_df.columns]
    
        ### Some analytics on each dataset ###
        #what column headers are in the DF?
        #columns_in_df = list(incoming_df.columns)
        #debug
        #print(columns_in_df)
        
        #see if there's some sort of column containing 'regid' substring
        #want to keep original datasets' naming conventions of each dataset for error handling/analysis
        id_cols = [col for col in incoming_df.columns if 'regid' in col]
        # #debug output
        # print(f'\n\033[1mParsing {source_name} ...\033[0m')
        # print(f'Found {len(id_cols)} columns in {source_name} containing ID info: {id_cols}')
    
        #find the SMILES column in each dataset
        smi_cols = [col for col in incoming_df.columns if 'smile' in col]
        #list_of_smiles_nums.append(num_of_smiles)
        # #debug output
        # print(f'Found {len(smi_cols)} columns in {source_name} containing SMILES info: {smi_cols}')
        
        #snag the columns of interest, SMILES and regid
        ids_and_parent_smiles = [(x,y) for x, y in zip(incoming_df['regid'], incoming_df['smiles'])]
        num_of_smiles = len(ids_and_parent_smiles)
        # print(f'# of Unique SMILES found in {source_name}: {num_of_smiles}')
        list_of_smiles_nums.append(num_of_smiles)
    
    #collect up the relevant data for visualization
    datasets_dataframe = pd.DataFrame(list(zip([csv.name for csv in csvs_to_check], list_of_smiles_nums)), columns=['Datasets', 'Num_SMILES'])
    #print current DF to console
    print(datasets_dataframe.head())

    #chop datasets down into the things I actually care about
    parsed_dataframes = []
    for csv_file in csvs_to_check:
        source_name = csv_file
        incoming_df = pd.read_csv(source_name)
        #make all the column headers lowercase for consistent parsing/finding correct columns
        incoming_df.columns = [x.lower() for x in incoming_df.columns]
    
        #capture first two columns of the imported CSV as tuple: (id, parent_smile)
        truncated_df = incoming_df[['regid', 'smiles']]
        #add the data source as a column in the front
        truncated_df.insert(loc=0, column='data_source', value=source_name)
        #debug
        #print(truncated_df.head())
    
        #save current truncated df to list of dfs
        parsed_dataframes.append(truncated_df)
    
    # #debug
    # print(f'\n\033[1mRetaining relevant data from each dataset...\033[0m')
    # for trunc_df in parsed_dataframes:
    #     print(trunc_df.head())
    #     print("")

    #combine individual truncated dfs into one megazord df
    dfs_to_combine = parsed_dataframes
    num_datasets = len(dfs_to_combine)
    
    #concat list of dfs into one df
    combined_df = pd.concat(dfs_to_combine, ignore_index=True)
    #how many raw SMILES in total df? (not deduplicated)
    len_non_deduplicated = len(combined_df)
    
    #debug
    print(f'\n\033[1mTotal combined SMILES across {num_datasets} datasets before deduplication:\033[0m {len_non_deduplicated}')
    #combined_df.head()

    return combined_df

In [134]:
def smiles_to_mols_rdkit(list_of_smiles: List[str], sanitization=True) -> List[Chem.Mol]:
    """
    Function: Converts an SD-file containing a single molecule to an RDKit mol object. Returns the mol object. 
    
    Input:
        - list_of_smiles: List[str]; a list of raw SMILES strings to convert to Mol objects
        - sanitization = True (default val True) - use RDKIT sanitization on the SMILES str; good/safe practice
        
    Returns:
        - list_of_valid_mols: List[Chem.Mol]; a list of validated RDKit Mol objects
    """

    #init a list to store validated RDKit mol objects
    list_of_valid_mols = []
    #Init a counter to keep track of any failed SMILES => Mol conversions
    failed_count = 0

    #loop over each of the raw SMILES strings in the incoming list    
    #Using 'enumerate' gives us access to the index of each SMILES in the list
    for index, smile in enumerate(list_of_smiles):
        
        try: #Attempt to convert the SMILES string to RDKit Mol Object 
            #here we attempt the conversion and explicitly require sanitization
            mol = Chem.MolFromSmiles(smile, sanitize = sanitization)

            if mol is not None: #If the mol is convertible (not NONE type), 
                #append the mol to our list of validated mols
                list_of_valid_mols.append(mol)

            else: #if the SMILE cant be converted, 
                failed_count += 1 #iterate our failed counter
                continue
                
        #some SMILES may be unconvertible to RDKIT Mols/catch some weird errors
        except Exception as e:
            failed_count += 1 #keep track of how many mols failed
            continue

    #if any mols failed, print how many 
    if failed_count >= 1:
        print(f"\nFailed to convert {failed_count} SMILES to RDKit Mols.")

    #give some feedback on the # of successfully converted Mols
    print(f"\nSucessfully converted {len(list_of_valid_mols)} SMILES to valid RDKit Mols.")

    #return the list of validated RDkit mols
    return list_of_valid_mols


In [135]:
def deduplicate_canonical_smiles(df_before_deduplication: pd.DataFrame) -> pd.DataFrame:
    """
    Function:  Deduplication assumes that all SMILES were previously canonicalized in "one pass" to 
    ensure that string-based deduplication is valid (this is critical to ensure successful
    deduplication as non-equivalent (aka potentially non-canonicalized) SMILES may represent
    the same molecule. 
    
    Input:
        - df_before_deduplication: pd.DataFrame; a pd DF containing valid chemical SMILES in a column with header 'smiles'
        
    Returns:
        - combined_df: pd.DataFrame; a deduplicated dataframe containing unique SMILES in a column
    """
    
    #Deduplicate based on Canonical SMILES string depuplication method (string-based)
    print(f"\033[1mDeduplicating incoming SMILES using Canonical SMILES deduplication... \033[0m")
    
    #pull smiles column from incoming DF into a list
    raw_list_of_smiles = df_before_deduplication.smiles
    
    ### Convert raw list of SMILES to a list of validated RDKit mol objects
    validated_mols = smiles_to_mols_rdkit(raw_list_of_smiles)

    #write Canonical SMILES from RDKit Mol objects
    canonical_smiles = [Chem.MolToSmiles(mol) for mol in validated_mols]

    #make a copy of the incoming DF for editing
    deduplicated_df = df_before_deduplication.copy()

    #replace the SMILES column with our new canonical SMILES
    deduplicated_df.smiles = canonical_smiles

    #drop duplicate rows based on String matching canonical SMILES
    deduplicated_df.drop_duplicates(subset=['smiles'], inplace=True)
    
    #check if anything was removed
    num_dedup = len(df_before_deduplication) - len(deduplicated_df)
    if num_dedup >= 1: #if any duplicates found, print how many
        print(f"Duplicate SMILES removed: {num_dedup}")

    print(f"\nNumber of SMILES after deduplication: {len(deduplicated_df)}")

    return deduplicated_df

In [154]:
def write_DF_to_csv(dataframe_to_write: pd.DataFrame, csv_name: str, output_directory: Path): 
    """
    Function: Write a Pandas DataFrame object to a .csv file in a specified directory.
    
    Input:
        - dataframe_to_write: pd.DataFrame; a Pandas DataFrame object to be written to .csv
        - csv_name: str; a string specifying what to call the 'output_name'.csv
        - output_directory: Path; a directory to write the .csv file to
        
    Returns: 
        N/A - nothing directly; writes a .csv file to the /output_directory. 
    """

    #validate that the dataframe contains some data
    if dataframe_to_write.empty: #uses pandas.empty method to check DF pop.
        raise ValueError("No data found in your DataFrame. Cannot write empty DataFrame to .csv")

    #ensure the output_directory exists; make it if it doesn't exist
    try:
        output_directory = Path(output_directory)
        output_directory.mkdir(parents=True, exist_ok=True)

    #Throw an exception error if 'output_directory' can't be created for some reason
    except Exception as e: 
        raise FileNotFoundError(f"Could not create output directory '/{output_directory.name}/': {e}")

    #format a dynamic filepath for the output .csv
    output_name = f"{csv_name}.csv"
    output_path = output_directory / output_name

    #Now try to write the file to the output_dir
    try:
        #write csv to target dir
        dataframe_to_write.to_csv(output_path, index=False)
        
        #confirm to user that output .csv file is written to local directory (ld)
        print(f"Successfully wrote '{csv_name}.csv' to '/{output_directory.name}/'")

    except Exception as e:
        raise IOError(f"Failed to write CSV file to {output_directory.name}: {e}")

In [155]:
#Parse a specified directory for .csv datasets containing SMILES data
combined_df_pre_deduplication = parse_csvs_for_smiles(input_csv_dir)

Parsing targeted '/input_csv_datasets/' for .csv files containing SMILES data...

                  Datasets  Num_SMILES
0  Sigman_ArBr_dataset.csv        5051
1   VEHICLe_10_dataset.csv       24867

Total combined SMILES across 2 datasets before deduplication: 29918


In [156]:
#Given the combined list of SMILES, deduplicate using Canonical SMILES Deduplication
deduplicated_df = deduplicate_canonical_smiles(combined_df_pre_deduplication)

Deduplicating incoming SMILES using Canonical SMILES deduplication... 

Sucessfully converted 29918 SMILES to valid RDKit Mols.

Number of SMILES after deduplication: 29918


In [157]:
write_DF_to_csv(deduplicated_df, 'combined_and_deduplicated_smiles', script_dir)

Successfully wrote 'combined_and_deduplicated_smiles.csv' to '/Module1_Merge_CSVs_Deduplicate_SMILES/'
